# KKBOX Churn Prediction — Advanced Model Training

**Improvements over lite version:**
- ✅ LightGBM model added
- ✅ Ensemble (XGBoost + LightGBM)
- ✅ StratifiedKFold 5-fold cross-validation
- ✅ Advanced transaction features
- ✅ Better hyperparameters

**Memory optimized:** Still skips user_logs to avoid kernel crashes

## Step 0 — Configuration & Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import log_loss, roc_auc_score, confusion_matrix, roc_curve

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import warnings
warnings.filterwarnings('ignore')

# Configuration
TARGET       = 'is_churn'
ID_COL       = 'msno'
TEST_SIZE    = 0.2
RANDOM_STATE = 42
N_FOLDS      = 5  # Cross-validation folds

pd.set_option('display.max_columns', None)
print('✅ All libraries loaded successfully.')

✅ All libraries loaded successfully.


## Step 1 — Load Data

In [2]:
print("Loading data...")
train        = pd.read_csv("../data/train_v2.csv")
members      = pd.read_csv("../data/members_v3.csv")
transactions = pd.read_csv("../data/transactions_v2.csv")

print(f"Train: {train.shape}, Members: {members.shape}, Transactions: {transactions.shape}")

Loading data...
Train: (970960, 2), Members: (6769473, 6), Transactions: (1431009, 9)


## Step 2 — Advanced Feature Engineering

In [3]:
# Clean members
members["bd"]     = members["bd"].clip(10, 80)
members["gender"] = members["gender"].fillna("unknown")

# Clean transactions
transactions = transactions.drop_duplicates()
transactions["actual_amount_paid"] = transactions["actual_amount_paid"].clip(0, 5000)
transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"].astype(str), format="%Y%m%d", errors="coerce")
transactions["membership_expire_date"] = pd.to_datetime(
    transactions["membership_expire_date"].astype(str), format="%Y%m%d", errors="coerce")

# Advanced transaction features
transactions["discount"]            = transactions["plan_list_price"] - transactions["actual_amount_paid"]
transactions["discount_rate"]        = transactions["discount"] / (transactions["plan_list_price"] + 1)
transactions["expiry_txn_interval"]  = (transactions["membership_expire_date"] - transactions["transaction_date"]).dt.days
transactions["amount_per_day"]       = transactions["actual_amount_paid"] / (transactions["payment_plan_days"] + 1)

# Basic aggregations
trans_agg = transactions.groupby("msno").agg({
    "actual_amount_paid": ["mean", "sum", "std"],
    "payment_plan_days":  ["mean", "std"],
    "is_cancel":          ["sum", "mean"],
    "is_auto_renew":      ["mean", "std"],
    "payment_method_id":  ["nunique"],
    "transaction_date":   ["count"]
})
trans_agg.columns = ["_".join(c) for c in trans_agg.columns]
trans_agg = trans_agg.reset_index().rename(columns={"transaction_date_count": "total_transactions"})

# Derived features
trans_agg["cancel_rate"]     = trans_agg["is_cancel_sum"] / (trans_agg["payment_plan_days_mean"] + 1)
trans_agg["payment_per_day"] = trans_agg["actual_amount_paid_sum"] / (trans_agg["payment_plan_days_mean"] + 1)

# Advanced aggregations
adv_agg = transactions.groupby("msno").agg({
    "discount":            ["mean", "sum", "max"],
    "discount_rate":       ["mean", "max"],
    "expiry_txn_interval": ["mean", "max", "min"],
    "amount_per_day":      ["mean", "std"]
})
adv_agg.columns = ["_".join(c) for c in adv_agg.columns]
adv_agg = adv_agg.reset_index()

# Last transaction features
last_txn = (
    transactions.sort_values("transaction_date")
    .groupby("msno").tail(1)
    [["msno","is_cancel","is_auto_renew","payment_plan_days",
      "actual_amount_paid","discount","expiry_txn_interval","payment_method_id"]]
    .rename(columns={
        "is_cancel":           "last_is_cancel",
        "is_auto_renew":       "last_is_auto_renew",
        "payment_plan_days":   "last_plan_days",
        "actual_amount_paid":  "last_amount_paid",
        "discount":            "last_discount",
        "expiry_txn_interval": "last_expiry_interval",
        "payment_method_id":   "last_payment_method"
    })
)

# Temporal features
REFERENCE_DATE = pd.to_datetime("2017-03-01")
last_trans = (
    transactions.sort_values("membership_expire_date")
    .groupby("msno").tail(1)
    [["msno", "membership_expire_date", "transaction_date"]]
)
last_trans["days_left"]             = (last_trans["membership_expire_date"] - REFERENCE_DATE).dt.days
last_trans["days_since_last_txn"]   = (REFERENCE_DATE - last_trans["transaction_date"]).dt.days
temporal_feat = last_trans[["msno", "days_left", "days_since_last_txn"]]

# Transaction recency features
txn_dates = transactions.groupby("msno")["transaction_date"].agg([
    ("first_txn_date", "min"),
    ("last_txn_date", "max")
]).reset_index()
txn_dates["customer_lifetime_days"] = (txn_dates["last_txn_date"] - txn_dates["first_txn_date"]).dt.days
txn_dates["days_since_first_txn"] = (REFERENCE_DATE - txn_dates["first_txn_date"]).dt.days
recency_feat = txn_dates[["msno", "customer_lifetime_days", "days_since_first_txn"]]

print("\n⚠️  SKIPPING user_logs features to avoid memory issues")

# Merge all
df = train.merge(members,      on="msno", how="left")
df = df.merge(trans_agg,        on="msno", how="left")
df = df.merge(adv_agg,          on="msno", how="left")
df = df.merge(last_txn,         on="msno", how="left")
df = df.merge(temporal_feat,    on="msno", how="left")
df = df.merge(recency_feat,     on="msno", how="left")

# Encode & fill
df = pd.get_dummies(df, columns=["city", "gender", "registered_via"], dummy_na=True)
df = df.fillna(0)

print(f"\n✅ Feature matrix: {df.shape[0]:,} rows x {df.shape[1]} cols")
print(f"Features: {df.shape[1]-2} (excl. msno, is_churn)")


⚠️  SKIPPING user_logs features to avoid memory issues

✅ Feature matrix: 970,960 rows x 70 cols
Features: 68 (excl. msno, is_churn)


## Step 3 — Train/Val Split

In [4]:
X = df.drop(columns=[TARGET, ID_COL])
y = df[TARGET]

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f'Training set   : {X_train.shape[0]:,} rows')
print(f'Validation set : {X_val.shape[0]:,} rows')
print(f'Features       : {X_train.shape[1]}')
print(f'\nChurn rate in train : {y_train.mean():.4f}')
print(f'Churn rate in val   : {y_val.mean():.4f}')

Training set   : 776,768 rows
Validation set : 194,192 rows
Features       : 68

Churn rate in train : 0.0899
Churn rate in val   : 0.0899


## Step 4 — Cross-Validation Training with Ensemble

Train both XGBoost and LightGBM using 5-fold cross-validation, then ensemble their predictions.

In [ ]:
# Calculate class weight
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count
print(f'scale_pos_weight = {scale_pos_weight:.2f}')

# Initialize StratifiedKFold
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# Store out-of-fold predictions
oof_xgb = np.zeros(len(X_train))
oof_lgb = np.zeros(len(X_train))

# Store validation predictions
val_xgb = np.zeros(len(X_val))
val_lgb = np.zeros(len(X_val))

print(f'\n{"="*60}')
print(f'Training with {N_FOLDS}-Fold Cross-Validation')
print(f'{"="*60}\n')

# Cross-validation training
for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train, y_train), 1):
    print(f'\n--- Fold {fold}/{N_FOLDS} ---')
    
    X_tr = X_train.iloc[train_idx]
    y_tr = y_train.iloc[train_idx]
    X_vl = X_train.iloc[valid_idx]
    y_vl = y_train.iloc[valid_idx]
    
    # XGBoost
    xgb = XGBClassifier(
        n_estimators     = 300,
        max_depth        = 7,
        learning_rate    = 0.03,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        scale_pos_weight = scale_pos_weight,
        eval_metric      = 'logloss',
        random_state     = RANDOM_STATE,
        n_jobs           = -1,
        verbosity        = 0
    )
    
    xgb.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=False)
    
    # Store out-of-fold predictions
    oof_xgb[valid_idx] = xgb.predict_proba(X_vl)[:, 1]
    # Average predictions on validation set
    val_xgb += xgb.predict_proba(X_val)[:, 1] / N_FOLDS
    
    xgb_fold_score = log_loss(y_vl, oof_xgb[valid_idx])
    print(f'XGBoost  Fold {fold} Log Loss: {xgb_fold_score:.4f}')
    
    # LightGBM
    lgb = LGBMClassifier(
        n_estimators     = 300,
        max_depth        = 7,
        learning_rate    = 0.03,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        scale_pos_weight = scale_pos_weight,
        random_state     = RANDOM_STATE,
        n_jobs           = -1,
        verbosity        = -1
    )
    
    lgb.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)])
    
    # Store out-of-fold predictions
    oof_lgb[valid_idx] = lgb.predict_proba(X_vl)[:, 1]
    # Average predictions on validation set
    val_lgb += lgb.predict_proba(X_val)[:, 1] / N_FOLDS
    
    lgb_fold_score = log_loss(y_vl, oof_lgb[valid_idx])
    print(f'LightGBM Fold {fold} Log Loss: {lgb_fold_score:.4f}')

# Calculate overall CV scores
xgb_cv_score = log_loss(y_train, oof_xgb)
lgb_cv_score = log_loss(y_train, oof_lgb)

print(f'\n{"="*60}')
print('Cross-Validation Results:')
print(f'{"="*60}')
print(f'XGBoost  CV Log Loss: {xgb_cv_score:.4f}')
print(f'LightGBM CV Log Loss: {lgb_cv_score:.4f}')

# Validation set scores
xgb_val_score = log_loss(y_val, val_xgb)
lgb_val_score = log_loss(y_val, val_lgb)
xgb_val_auc = roc_auc_score(y_val, val_xgb)
lgb_val_auc = roc_auc_score(y_val, val_lgb)

print(f'\nValidation Set Results:')
print(f'XGBoost  - Log Loss: {xgb_val_score:.4f}, ROC-AUC: {xgb_val_auc:.4f}')
print(f'LightGBM - Log Loss: {lgb_val_score:.4f}, ROC-AUC: {lgb_val_auc:.4f}')

# Ensemble predictions (simple average)
ensemble_val = (val_xgb + val_lgb) / 2
ensemble_score = log_loss(y_val, ensemble_val)
ensemble_auc = roc_auc_score(y_val, ensemble_val)

print(f'\n🎯 ENSEMBLE - Log Loss: {ensemble_score:.4f}, ROC-AUC: {ensemble_auc:.4f}')
print(f'{"="*60}')

scale_pos_weight = 10.12

Training with 5-Fold Cross-Validation


--- Fold 1/5 ---


## Step 5 — Train Final Models on Full Data

In [ ]:
print("Training final models on full training data...\n")

# Combine train and validation for final training
X_full = pd.concat([X_train, X_val])
y_full = pd.concat([y_train, y_val])

# Train final XGBoost
final_xgb = XGBClassifier(
    n_estimators     = 300,
    max_depth        = 7,
    learning_rate    = 0.03,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    scale_pos_weight = scale_pos_weight,
    eval_metric      = 'logloss',
    random_state     = RANDOM_STATE,
    n_jobs           = -1,
    verbosity        = 0
)
final_xgb.fit(X_full, y_full)
print("✓ XGBoost trained")

# Train final LightGBM
final_lgb = LGBMClassifier(
    n_estimators     = 300,
    max_depth        = 7,
    learning_rate    = 0.03,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    scale_pos_weight = scale_pos_weight,
    random_state     = RANDOM_STATE,
    n_jobs           = -1,
    verbosity        = -1
)
final_lgb.fit(X_full, y_full)
print("✓ LightGBM trained")

print("\n✅ Final models ready for prediction")

## Step 6 — Generate Ensemble Submission

In [ ]:
test = pd.read_csv("../data/sample_submission_v2.csv")
print(f"Test set shape: {test.shape}")

# Apply same preprocessing
df_test = test[["msno"]].merge(members,   on="msno", how="left")
df_test = df_test.merge(trans_agg,         on="msno", how="left")
df_test = df_test.merge(adv_agg,           on="msno", how="left")
df_test = df_test.merge(last_txn,          on="msno", how="left")
df_test = df_test.merge(temporal_feat,     on="msno", how="left")
df_test = df_test.merge(recency_feat,      on="msno", how="left")

df_test = pd.get_dummies(df_test, columns=["city", "gender", "registered_via"], dummy_na=True)
df_test = df_test.fillna(0)

# Align columns
train_cols = X_train.columns.tolist()
for col in train_cols:
    if col not in df_test.columns:
        df_test[col] = 0
X_test = df_test[train_cols]

print(f"Test feature matrix: {X_test.shape}")

# Predict with both models
test_xgb = final_xgb.predict_proba(X_test)[:, 1]
test_lgb = final_lgb.predict_proba(X_test)[:, 1]

# Ensemble (simple average)
test_ensemble = (test_xgb + test_lgb) / 2

submission = pd.DataFrame({
    "msno":     test["msno"],
    "is_churn": test_ensemble
})

print(f"\nSubmission shape: {submission.shape}")
print(f"Predicted churn rate: {test_ensemble.mean():.4f}")
print(f"  - XGBoost:  {test_xgb.mean():.4f}")
print(f"  - LightGBM: {test_lgb.mean():.4f}")
display(submission.head())

submission.to_csv("../data/submission_advanced.csv", index=False)
print("\n✅ Saved to ../data/submission_advanced.csv")

## Step 7 — Feature Importance Analysis

In [ ]:
# Get feature importance from both models
xgb_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': final_xgb.feature_importances_
}).sort_values('importance', ascending=False)

lgb_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': final_lgb.feature_importances_
}).sort_values('importance', ascending=False)

# Plot top 20 features
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# XGBoost
axes[0].barh(xgb_importance.head(20)['feature'], xgb_importance.head(20)['importance'])
axes[0].set_xlabel('Importance')
axes[0].set_title('XGBoost - Top 20 Features')
axes[0].invert_yaxis()

# LightGBM
axes[1].barh(lgb_importance.head(20)['feature'], lgb_importance.head(20)['importance'])
axes[1].set_xlabel('Importance')
axes[1].set_title('LightGBM - Top 20 Features')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print("\nTop 10 Features (XGBoost):")
print(xgb_importance.head(10))

print("\nTop 10 Features (LightGBM):")
print(lgb_importance.head(10))